# 🪙 cyandle: crypto transformer training

train the cryptocurrency transformer model on colab t4 gpu.

## setup

In [ ]:
# colab settings
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# mount drive if needed
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# install dependencies
%%bash
pip install torch pandas numpy pyarrow scikit-learn lightgbm --quiet

In [ ]:
# clone repo
%%bash
git clone https://github.com/mukundzha/cyandle.git
cd cyandle && pip install -e . --quiet

In [ ]:
# verify gpu
import torch
print(f'device: {torch.device("cuda" if torch.cuda.is_available() else "cpu")}')
if torch.cuda.is_available():
    print(f'gpu: {torch.cuda.get_device_name(0)}')
    print(f'vram: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} gb')

In [ ]:
# quick test data generation
import pandas as pd
import numpy as np

dates = pd.date_range('2024-01-01', periods=5000, freq='5min')
df = pd.DataFrame({
    'open_time': dates,
    'open': 50000 + np.random.randn(5000).cumsum() * 100,
    'high': 50000 + np.random.randn(5000).cumsum() * 100 + 500,
    'low': 50000 + np.random.randn(5000).cumsum() * 100 - 500,
    'close': 50000 + np.random.randn(5000).cumsum() * 100,
    'volume': 1000 + np.abs(np.random.randn(5000).cumsum()) * 10,
})

df.to_parquet('sample_data.parquet')
print(f'created {len(df)} candlesticks')

In [ ]:
# dry run - test forward/backward
%cd cyandle
!python scripts/train.py --data ../sample_data.parquet --dry-run

In [ ]:
# full training
%cd cyandle
!python scripts/train.py \
    --data ../sample_data.parquet \
    --model medium \
    --epochs 5 \
    --batch-size 32

## results

check the `cryptoformer/experiments/` folder for outputs.